# Dual-Branch Volatility Network: Training Pipeline

This notebook handles the offline training phase for our State-Switching Volatility Network. 
The pipeline consists of:
1. Fetching historical 5-minute OHLCV data.
2. Engineering structural features (Variance Ratio, MFI, Volume Acceleration).
3. Formatting sequences into 78-bar lookback windows for PyTorch.
4. Training the CNN-LSTM Dual-Branch architecture using QLIKE Loss.
5. Saving the static weights for live execution.

In [1]:
import sys
import os
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add the project root to the system path to import our custom src modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.features import FeatureEngineer
from src.architecture import DualBranchVolatilityNet, QLIKELoss

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Define paths to your local raw data
# Assuming the notebook is running from the 'notebooks/' directory
train_csv_path = "../data/raw/ETHUSDT_5m_2024-01-01_to_2026-01-01.csv"
test_csv_path = "../data/raw/ETHUSDT_5m_2026-01-01_to_2026-06-01.csv"

print("Loading local raw CSV files...")

# Load training data
train_df = pd.read_csv(train_csv_path)
# Ensure timestamp is a datetime object and set as index for the FeatureEngineer
train_df['open_time'] = pd.to_datetime(train_df['open_time'])
train_df.set_index('open_time', inplace=True)

# Load testing data
test_df = pd.read_csv(test_csv_path)
test_df['open_time'] = pd.to_datetime(test_df['open_time'])
test_df.set_index('open_time', inplace=True)

print(f"Training set size: {train_df.shape[0]} rows")
print(f"Testing set size: {test_df.shape[0]} rows")

Loading local raw CSV files...
Training set size: 210529 rows
Testing set size: 43489 rows


In [6]:
# Initialize Feature Engineer with 78-bar rolling window
engineer = FeatureEngineer(vr_q=5, rolling_window=78)

print("Processing training features...")
train_processed = engineer.build_feature_set(train_df)

print("Processing testing features...")
test_processed = engineer.build_feature_set(test_df)

# Define feature columns
feature_cols = [
    'open', 'high', 'low', 'close', 'volume_usdt', 
    'sin_time', 'cos_time', 'variance_ratio', 
    'mfi', 'vol_acceleration'
]

# Calculate target (Realized Volatility proxy) and clean infinite values
for df in [train_processed, test_processed]:
    df['target_rv'] = (np.log(df['close'].shift(-1) / df['close']))**2
    df['target_rv'] = df['target_rv'].clip(lower=1e-8)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

print(f"Final Train Sequences: {train_processed.shape[0]}")
print(f"Final Test Sequences: {test_processed.shape[0]}")

# ==========================================
# Export Processed Data
# ==========================================
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

train_processed_path = os.path.join(processed_dir, "train_processed.csv")
test_processed_path = os.path.join(processed_dir, "test_processed.csv")

train_processed.to_csv(train_processed_path)
test_processed.to_csv(test_processed_path)

print(f"\nSuccessfully saved processed training data to: {train_processed_path}")
print(f"Successfully saved processed testing data to: {test_processed_path}")

Processing training features...
Processing testing features...
Final Train Sequences: 210527
Final Test Sequences: 43487

Successfully saved processed training data to: ../data/processed/train_processed.csv
Successfully saved processed testing data to: ../data/processed/test_processed.csv


In [7]:
# ==========================================
# Cell 5: PyTorch Dataset Preparation
# ==========================================
class SequenceDataset(Dataset):
    """
    Creates overlapping 78-bar sequences to feed into the CNN/LSTM.
    """
    def __init__(self, dataframe, features, target, seq_len=78):
        self.features = dataframe[features].values
        self.target = dataframe[target].values
        self.seq_len = seq_len

    def __len__(self):
        return len(self.features) - self.seq_len

    def __getitem__(self, idx):
        # Grab the sequence of length 'seq_len'
        x = self.features[idx : idx + self.seq_len]
        # Target is the volatility immediately following the sequence
        y = self.target[idx + self.seq_len]
        
        return torch.tensor(x, dtype=torch.float32), torch.tensor([y], dtype=torch.float32)

SEQ_LEN = 78

# 1. Create separate PyTorch Dataset instances for Train and Test
train_dataset = SequenceDataset(train_processed, feature_cols, 'target_rv', seq_len=SEQ_LEN)
test_dataset = SequenceDataset(test_processed, feature_cols, 'target_rv', seq_len=SEQ_LEN)

# 2. Initialize DataLoaders
# Train loader shuffles to break local correlations and stabilize gradients
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Test/Validation loader remains sequentially ordered (shuffle=False)
val_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Total Training Sequences: {len(train_dataset)} | Batches: {len(train_loader)}")
print(f"Total Testing Sequences: {len(test_dataset)} | Batches: {len(val_loader)}")

Total Training Sequences: 210449 | Batches: 3289
Total Testing Sequences: 43409 | Batches: 679


In [8]:
# Instantiate the Dual-Branch Network
model = DualBranchVolatilityNet(
    num_features=len(feature_cols), 
    seq_len=SEQ_LEN, 
    hidden_dim=64
).to(device)

# Using our custom QLIKE Loss to severely penalize underpredicted risk
criterion = QLIKELoss()

# Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

print(model)

DualBranchVolatilityNet(
  (conv1): Conv1d(10, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (cnn_fc): Linear(in_features=1216, out_features=64, bias=True)
  (lstm): LSTM(10, 64, num_layers=2, batch_first=True, dropout=0.2)
  (attention_gate): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
    (3): Sigmoid()
  )
  (fc_out): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
    (3): Softplus(beta=1.0, threshold=20.0)
  )
)


In [9]:
epochs = 10
train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass (Note: Model returns both prediction and gate weights)
        predictions, gate_weights = model(batch_x)
        
        # Compute QLIKE loss
        loss = criterion(predictions, batch_y)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            predictions, _ = model(batch_x)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train QLIKE Loss: {avg_train_loss:.4f} | Val QLIKE Loss: {avg_val_loss:.4f}")

# Plot training curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title("QLIKE Loss over Epochs")
plt.legend()
plt.show()

KeyboardInterrupt: 

In [ ]:
# Create the models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the model state dictionary 
# These weights will remain static throughout the live execution phase
model_save_path = '../models/dual_branch_weights.pth'
torch.save(model.state_dict(), model_save_path)

print(f"Training complete. Static model weights successfully saved to: {model_save_path}")